# Task 2B: Winning Classification Algorithms

In this notebook, we analyze the winning approach of a major classification competition from Kaggle. We selected the **Otto Group Product Classification Challenge (2015)** because it closely mirrors our own work -- tabular data, multi-class classification, and a strong showing by XGBoost-based methods.

## 1. Competition Description

The **Otto Group Product Classification Challenge** was hosted on Kaggle in 2015 by the Otto Group, one of the world's largest e-commerce companies. The goal was to classify products into **9 distinct categories** based on 93 anonymized numerical features. The dataset contained **61,878 labeled training samples** and **144,368 unlabeled test samples**. A total of **3,514 teams** participated, making it one of the most popular Kaggle competitions at the time.

The evaluation metric was **multi-class logarithmic loss (log-loss)**, which penalizes confident but incorrect predictions heavily. This metric encourages well-calibrated probability estimates rather than hard class assignments. The anonymized nature of the features meant that domain knowledge could not be leveraged -- success depended entirely on algorithmic and ensemble strategies.

The competition attracted significant attention because it represented a clean benchmark for comparing different machine learning paradigms (tree-based methods, neural networks, nearest neighbors) on a moderately sized tabular dataset. Many top solutions were later published in detail, making this competition a valuable case study for understanding state-of-the-art classification techniques.

## 2. The Winning Approach

The 1st place solution was submitted by the team **"tin rtyz"** (a collaboration between top Kaggle competitors). Their approach centered on a **weighted ensemble of approximately 30 diverse models**, organized in a sophisticated **three-layer stacking architecture**.

### Base Models (Layer 1)

The first layer consisted of a diverse set of individual models, each trained using stratified k-fold cross-validation (typically 5 or 10 folds). The base model types included:

- **XGBoost** -- trained with the `multi:softprob` objective to output calibrated class probabilities. Multiple XGBoost models were trained with different hyperparameter configurations (tree depth, learning rate, subsampling ratios) to increase diversity.
- **Neural Networks (Keras)** -- feedforward networks with 2-3 hidden layers, using dropout (0.5) as the primary regularization technique. These were **bagged** (trained on different bootstrap samples) to reduce variance. The neural networks captured non-linear feature interactions that tree-based methods sometimes miss.
- **K-Nearest Neighbors (KNN)** -- distance-based classifiers that contributed a fundamentally different type of prediction. KNN predictions also served as **meta-features** for higher layers.
- **Random Forests and Extra Trees** -- additional tree-based models with different splitting strategies to complement XGBoost.

### Stacking Architecture (Layers 2-3)

The out-of-fold predictions from Layer 1 models were used as input features for Layer 2 meta-learners. Layer 2 itself was stacked into a Layer 3 meta-learner (typically a simple logistic regression or XGBoost with conservative parameters). This multi-level stacking allowed the ensemble to learn complex relationships between base model predictions.

### Key Innovations

- **Model diversity over model quality**: rather than perfecting a single model, the team invested in creating diverse models that made different types of errors.
- **Bagged neural networks**: training multiple neural networks on bootstrap samples and averaging predictions significantly reduced the variance inherent in neural network training.
- **KNN as meta-features**: KNN predictions captured local neighborhood structure that other models could not easily learn, providing unique signal to the stacker.
- **Stratified cross-validation**: ensured that each fold maintained the class distribution, critical for the log-loss metric.

## 3. Why This Approach Won

The winning solution's success can be attributed to several key factors:

**Diversity of base models** -- by combining fundamentally different learning paradigms (gradient boosting, neural networks, instance-based learning, random forests), the ensemble captured different aspects of the data. Where XGBoost excels at axis-aligned decision boundaries, neural networks capture smooth non-linear interactions, and KNN captures local neighborhood patterns.

**Stacking learns optimal combination** -- rather than using simple averaging, the stacking meta-learner learned data-driven weights for combining base models. This is strictly superior to uniform or manually-tuned weighting, as it can adapt to the strengths and weaknesses of each base model across different regions of the feature space.

**Calibrated probabilities** -- since the metric was log-loss, having well-calibrated probability outputs was essential. XGBoost with softprob and neural networks with softmax naturally produce probability estimates, and the stacking procedure further improved calibration.

### Performance Comparison

| Approach | Approximate Leaderboard Rank | Log-Loss |
|----------|------------------------------|----------|
| Single XGBoost (well-tuned) | Top 10% (~350th) | ~0.44 |
| Single Neural Network | Top 15% (~500th) | ~0.46 |
| Simple average of XGB + NN | Top 3% (~100th) | ~0.41 |
| 3-layer stacked ensemble (winner) | 1st place | ~0.38 |

The table above illustrates the diminishing returns but consistent improvement from increasingly sophisticated ensembling. A single well-tuned XGBoost model already achieves strong performance, but the gap between that and the winning solution is substantial in terms of ranking. The jump from simple averaging to full stacking demonstrates the value of learned model combination.

## 4. Relevance to Our Work

The Otto competition results closely parallel several findings from our own mood prediction task:

- **XGBoost as the strongest single model**: In both the Otto competition and our work, XGBoost consistently outperformed other individual classifiers. This reinforces XGBoost's reputation as the go-to algorithm for tabular classification tasks, thanks to its built-in regularization and efficient handling of feature interactions.

- **Stacking results diverge due to dataset size**: The Otto winner's 3-layer stacking worked because the dataset had 61,878 samples -- enough to train meta-learners without overfitting. In our work (iteration 05), stacking actually **hurt** performance because our dataset contains only ~1,965 samples. With limited data, the meta-learner overfits to the out-of-fold predictions of the base models rather than learning genuine combination patterns.

- **Neural networks as complementary models**: The winner used neural networks alongside XGBoost to capture different patterns. Similarly, our pipeline explored GRU-based recurrent networks for temporal patterns alongside XGBoost for static features. However, the winner had sufficient data to train deep networks effectively, while our small sample size limited neural network performance.

- **Key lesson about ensemble methods**: The Otto competition demonstrates that ensemble complexity should scale with dataset size. For large tabular datasets (50k+ samples), multi-layer stacking with diverse models is the dominant strategy. For small datasets like ours, a single well-tuned model (or at most a simple weighted average) is more appropriate to avoid overfitting.

## 5. References

1. **Otto Group Product Classification Challenge** -- Kaggle Competition Page. https://www.kaggle.com/c/otto-group-product-classification-challenge

2. **1st Place Solution Write-up** -- "tin rtyz" team solution. https://www.kaggle.com/c/otto-group-product-classification-challenge/discussion/14335

3. Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785-794.

4. Wolpert, D. H. (1992). Stacked Generalization. *Neural Networks*, 5(2), 241-259.